Import Statements

In [84]:
import pandas as pd
from datetime import datetime,timedelta
from typing import Dict, List, Union, Final
import numpy as np

Configuration and Inputs

In [85]:

START_DATE: Final[str] = datetime(2026,1,1).date()
lot_size=550

temp=pd.read_csv('Feb_FUT.csv')
Fut=temp[['Date','Settlement Price']].copy()
temp2=pd.read_csv('Mar_FUT.csv')
Mar_Fut=temp2[['Date','Settlement Price']].copy()

Making the date into indexes for the settlement prices

In [86]:
if 'Date' in Fut.columns:
 Fut['Date'] = pd.to_datetime(Fut['Date'], dayfirst=True)
 Fut['Settlement Price'] = pd.to_numeric(Fut['Settlement Price'].astype(str).str.replace(',', ''),errors='coerce')
 Fut.set_index('Date', inplace=True)
 Fut.sort_index(inplace=True)
 Fut=Fut.loc['2026-01']


if 'Date' in Mar_Fut.columns:
 Mar_Fut['Date'] = pd.to_datetime(Mar_Fut['Date'], dayfirst=True)
 Mar_Fut['Settlement Price'] = pd.to_numeric(Mar_Fut['Settlement Price'].astype(str).str.replace(',', ''),errors='coerce')
 Mar_Fut.set_index('Date', inplace=True)
 Mar_Fut.sort_index(inplace=True)
 Mar_Fut=Mar_Fut.loc['2026-01']

Creating Accounts Dataframe

In [87]:
Feb_Accounts =pd.DataFrame(columns=["Date", "Margin_A/C", "Borrowing_A/C", "Interest_Due"])

if 'Date' in Feb_Accounts.columns:
 Feb_Accounts['Date'] = pd.to_datetime(Feb_Accounts['Date'], dayfirst=True)
 Feb_Accounts.set_index('Date', inplace=True)




F0=Fut.loc['2026-01-01', 'Settlement Price']

Contract_Price=F0*lot_size
Num_Contracts=(round(15000000/Contract_Price))

Money_used=0.3*(Contract_Price)*Num_Contracts

Feb_Accounts.loc[START_DATE]=(Money_used/100000,0.0,0.0)

Money_left=5000000.0-Money_used
maintenance_margin=Money_used*(2/3)

Borrowed=0.0
Interest=0.0
rate=0.0985

In [ ]:
P0 = F0
initial_margin_requirement = Money_used
rate = 0.0985 / 365

skip_date=pd.Timestamp(2026,1,1).date()
Last_Day=pd.to_datetime(skip_date)

for i in Fut.index:
    clean = i.date()
    if clean == skip_date:
        continue
    
    P1 = Fut.at[i, 'Settlement Price']
    
    i=pd.to_datetime(i)
    t=(i-Last_Day).days

    daily_interest = Borrowed * (pow((1+rate),t/365)-1)
    Interest += daily_interest
    
    # 2. Daily MTM Profit/Loss
    change = (P1 - P0) * Num_Contracts * lot_size
    Money_used += change
    
   
            
    P0 = P1
    Last_Day=i
    
    # Save results in Lakhs (dividing by 100,000)
    Feb_Accounts.loc[clean] = (
        Money_used / 100000, 
        Borrowed / 100000, 
        Interest 
    )
     # 3. Handle Margin Calls
    if Money_used < maintenance_margin:
        top_up = initial_margin_requirement - Money_used
        Money_used += top_up
        
        # Pulling from the bank
        if Money_left >= top_up:
            Money_left -= top_up
        else:
            # Use what's left in cash, then add the rest to debt
            needed_from_loan = top_up - max(0, Money_left)
            Money_left = 0 
            Borrowed += needed_from_loan



In [89]:
Mar_Accounts =pd.DataFrame(columns=["Date", "Margin_A/C", "Borrowing_A/C", "Interest_Due"])

if 'Date' in Mar_Accounts.columns:
 Mar_Accounts['Date'] = pd.to_datetime(Mar_Accounts['Date'], dayfirst=True)
 Mar_Accounts.set_index('Date', inplace=True)




F0=Mar_Fut.loc['2026-01-01', 'Settlement Price']

Contract_Price=F0*lot_size
Num_Contracts=(round(15000000/Contract_Price))

Money_used=0.3*(Contract_Price)*Num_Contracts

Mar_Accounts.loc[START_DATE]=(Money_used/100000,0.0,0.0)

Money_left=5000000.0-Money_used
maintenance_margin=Money_used*(2/3)

Borrowed=0.0
Interest=0.0
rate=0.0985

In [ ]:
P0 = F0
initial_margin_requirement = Money_used
rate = 0.0985 / 365

skip_date=pd.Timestamp(2026,1,1).date()
Last_Day=pd.to_datetime(skip_date)

for i in Mar_Fut.index:
    clean = i.date()
    if clean == skip_date:
        continue
    
    P1 = Mar_Fut.at[i, 'Settlement Price']
    
    i=pd.to_datetime(i)
    t=(i-Last_Day).days

    daily_interest = Borrowed * (pow((1+rate),t/365)-1)
    Interest += daily_interest
    
    # 2. Daily MTM Profit/Loss
    change = (P1 - P0) * Num_Contracts * lot_size
    Money_used += change
    
   
            
    P0 = P1
    Last_Day=i
    
    # Save results in Lakhs (dividing by 100,000)
    Mar_Accounts.loc[clean] = (
        Money_used / 100000, 
        Borrowed / 100000, 
        Interest 
    )
     # 3. Handle Margin Calls
    if Money_used < maintenance_margin:
        top_up = initial_margin_requirement - Money_used
        Money_used += top_up
        
        # Pulling from the bank
        if Money_left >= top_up:
            Money_left -= top_up
        else:
            # Use what's left in cash, then add the rest to debt
            needed_from_loan = top_up - max(0, Money_left)
            Money_left = 0 
            Borrowed += needed_from_loan


# Assuming your dataframes are df1 and df2
with pd.ExcelWriter('Futures.xlsx') as writer:
    Feb_Accounts.to_excel(writer, sheet_name='Febuary Futures')
    Mar_Accounts.to_excel(writer, sheet_name='March Futures')
